This notebook is meant to be run AFTER all of the filled-out `labels.csv` files are added to the necessary label_sesion directory. Running this notebook will combine the new labels with all of the old labels and data

In [11]:
from pathlib import Path
import pandas as pd
import numpy as np
import ast # turns str(list(int)) to list(int)
import matplotlib.pyplot as plt
import json

In [27]:
session_dir = Path('/data/vision/polina/users/marcusbl/bin_class/label_sessions_data/label_session_3-11')

old_data = pd.read_csv(session_dir / 'labeled_slice_df.csv', index_col = 0)

# Create DataFrame for NEW labels

This dataframe will have following columns:
- `path`: path to the stack 
- `slice_num`: which slice in the stack we are referring to
- `labels`: list of all labels [0 = good, 1 = bad, 2 = questionable]
- `is_edge`: list of boolean Flag whether this is an edge slice or not

In [33]:
# lookup: (pdf_num, stack_num) -> stack
with open(session_dir / 'lookup.json', 'r') as f:
    stack_lookup = json.load(f)

In [87]:
def get_label(slice_num: int, bad_slices: set[int], quest_slices: set[int]):
    """
    Returns the label for a given slice, given the bad & questionable slices in the stack

    GOOD = 0; BAD = 1; QUESTIONABLE = 2
    """
    if slice_num in bad_slices:
        return 1
    elif slice_num in quest_slices:
        return 2
    else:
        return 0

In [88]:
results = []

In [89]:
for group_dir in session_dir.iterdir():

    # skip the files in the session_dir
    if not group_dir.is_dir(): 
        continue 

    pdf_num = int(group_dir.stem.split('group')[-1])
    all_stack_info = pd.read_csv(group_dir / f'labels_{pdf_num}.csv', index_col = 0) 

    # Loop through all stacks in this pdf
    for stack_num in all_stack_info.columns:
        stack_info = all_stack_info[stack_num]

        skip_column = stack_info.iloc[0] is np.nan

        if skip_column:
            # print(f"No start/end info found for {pdf_num}-{stack_num}. Skipping...")
            continue

        # Get start and end for non-edges
        start = int(stack_info.iloc[0])
        end = int(stack_info.iloc[1])

        # Get stack path & which slices are valid
        lookup_key = f"{pdf_num}-{stack_num}"
        if lookup_key not in stack_lookup:
            continue
        
        stack_path = stack_lookup[lookup_key]
        valid_slices_str = old_data.loc[old_data['path'] == str(stack_path), 'stack_slices'].iloc[0]
        valid_slices = ast.literal_eval(valid_slices_str) # str -> int

        # Get labels for the stack
        stack_label_info = stack_info.iloc[3:] # skip first 3 rows
        is_quest = stack_label_info.astype(str).str.contains('\?')
        bad_slices = set(int(num) for num in stack_label_info[~is_quest] if num is not np.nan)
        quest_slices = set(int(num[:-1]) for num in stack_label_info[is_quest] if num is not np.nan)
            
        #Add each slice to results
        for slice_num in valid_slices:
            results.append({
                'path': stack_path, 
                'slice_num': slice_num, 
                'label': get_label(slice_num, bad_slices, quest_slices),
                'is_edge': (slice_num < start) or (slice_num > end),
                'pdf_num': pdf_num
            })